# 09 · Pose vs scene

The same linear probe and the same 50 session-aware folds (10 repeats of 5-fold CV), on three versions of every clip encoded by V-JEPA 2:
- **full frame** (notebook 03);
- **surfer-only**: the clip cut to the surfer region (notebook 08);
- **scene-only**: the surfer region painted gray (notebook 08).

If a class keeps its score on surfer-only clips, the model relies on the surfer (pose and movement). If it keeps its score on scene-only clips, the scene alone carries the answer, which would be a shortcut.

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Make the project root importable (notebooks/ is one level below it)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.data.features import load_features
from src.models.evaluate import per_class_f1_cv

splits = pd.read_csv(PROJECT_ROOT / "data" / "splits.csv")
y = splits["maneuver"].to_numpy()
groups = splits["group"].to_numpy()

# Mean over the 8 time slices, as for the linear probe in notebook 04
versions = {
    "full frame": load_features(splits, "vjepa2_vitl_t8").mean(axis=1),
    "surfer-only": load_features(splits, "vjepa2_vitl_t8_surfer").mean(axis=1),
    "scene-only": load_features(splits, "vjepa2_vitl_t8_scene").mean(axis=1),
}
for name, X in versions.items():
    print(f"{name:12s} {X.shape}")

full frame   (1056, 1024)
surfer-only  (1056, 1024)
scene-only   (1056, 1024)


## 2. Per-class F1 by clip version

In [2]:
# Per-class F1 (mean over 10 repeats) for each version, with macro F1 as the average of the classes
scores = {}
for name, X in versions.items():
    f1 = per_class_f1_cv(X, y, groups)
    scores[name] = f1.assign(macro=f1.mean(axis=1))
    print(f"{name} done")

mean = pd.DataFrame({name: s.mean() for name, s in scores.items()}).round(3)
std = pd.DataFrame({name: s.std() for name, s in scores.items()}).round(3)

# Change against the full frame, in F1 points
delta = (mean[["surfer-only", "scene-only"]].sub(mean["full frame"], axis=0) * 100).round(1)
delta.columns = [f"{c} vs full (pts)" for c in delta.columns]

print("\nMean F1 over 10 repeats:")
print(mean)
print("\nStd over repeats:")
print(std)
print("\nChange against the full frame:")
print(delta)

full frame done
surfer-only done
scene-only done

Mean F1 over 10 repeats:
                   full frame  surfer-only  scene-only
360                     0.851        0.872       0.601
cutback-frontside       0.836        0.848       0.628
roller                  0.891        0.908       0.748
take-off                0.944        0.955       0.741
macro                   0.880        0.896       0.680

Std over repeats:
                   full frame  surfer-only  scene-only
360                     0.017        0.012       0.018
cutback-frontside       0.010        0.013       0.013
roller                  0.007        0.006       0.006
take-off                0.014        0.003       0.027
macro                   0.007        0.006       0.011

Change against the full frame:
                   surfer-only vs full (pts)  scene-only vs full (pts)
360                                      2.1                     -25.0
cutback-frontside                        1.2                     -20.8
r

## 3. Findings

- **The probe relies on the surfer.** On surfer-only clips, macro F1 rises from 0.880 (full frame) to 0.896, and every class improves (+1.1 to +2.1 points). The gain is larger than the spread across repeats (std about 0.006). A likely reason: the encoder resizes its input to 256 x 256, so the crop gives the surfer many more pixels and removes distracting water.
- **Without the surfer, every class drops.** On scene-only clips, macro F1 falls to 0.680 (−20 points), with drops of 14 to 25 points per class.
- **Take-off is not a scene shortcut.** It loses 20.3 points without the surfer, like the other classes, and reaches 0.955 ± 0.003 on surfer-only clips. Together with notebook 07, the evidence points to the take-off posture.
- **Scene-only is an upper bound on scene cues, not a pure scene test.** The masked clips still carry the size and position of the gray region (which follows the surfer's path), spray outside the margin, camera motion that tracks the surfer, and the shape of the wave. Even so, they stay far below the surfer-only clips.
- **Consequence.** The long-video timeline and the demo app use surfer-only clips (detect, crop, encode, classify).